In [2]:
pip install selenium pandas webdriver-manager



   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.7 MB 4.4 MB/s eta 0:00:03
   ---------- ----------------------------- 2.6/9.7 MB 7.5 MB/s eta 0:00:01
   --------------------- ------------------ 5.2/9.7 MB 9.3 MB/s eta 0:00:01
   ---------------------- ----------------- 5.5/9.7 MB 9.4 MB/s eta 0:00:01
   ---------------------- ----------------- 5.5/9.7 MB 9.4 MB/s eta 0:00:01
   ---------------------- ----------------- 5.5/9.7 MB 9.4 MB/s eta 0:00:01
   ----------------------- ---------------- 5.8/9.7 MB 4.5 MB/s eta 0:00:01
   -------------------------- ------------- 6.3/9.7 MB 3.8 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.7 MB 3.8 MB/s eta 0:00:01
   --------------------------------- ------ 8.1/9.7 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.7 MB 4.2 MB/s eta 0:00:01
   -----------------------

In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


url = "https://edv.scsc.gov.et/ICSMISCensus"

headers = {
     "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}



import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Target URL
url = "https://edv.ecsc.gov.et/ICSMISCensus"

# Add a User-Agent header to make the request look like a normal browser visit
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

try:
    # 1. Send HTTP GET request
    response = requests.get(url, headers=headers, timeout=15, verify=True)
    
    # Check if the page loaded successfully
    if response.status_code == 200:
        # 2. Parse HTML content
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Create a list to store extracted records
        scraped_data = []
        
        # --- ADJUST SELECTORS BASED ON SITE HTML ---
        # Look for table rows, list items, or divs containing the records
        # For example, if it's a standard HTML table:
        table_rows = soup.find_all('tr') 
        
        for row in table_rows:
            cells = row.find_all('td')
            if cells:
                # Extract text clean of extra spaces/newlines
                row_data = [cell.get_text(strip=True) for cell in cells]
                scraped_data.append(row_data)
        
        # 3. Save data to a clean format
        if scraped_data:
            df = pd.DataFrame(scraped_data)
            df.to_csv("ecsc_census_data.csv", index=False)
            print(f"Success! Saved {len(scraped_data)} rows to 'ecsc_census_data.csv'")
        else:
            print("Connected, but no items found matching the HTML selectors. Inspect the page elements.")
            
    else:
        print(f"Failed to connect. Status Code: {response.status_code}")

except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")


Connected, but no items found matching the HTML selectors. Inspect the page elements.


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# Target URL
url = "https://ecsc.gov.et"

# Set up headless Chrome options (runs in background)
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Remove this line if you want to visually watch the browser open
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize Browser Driver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

try:
    print("Opening the page... please wait.")
    driver.get(url)
    
    # 1. WAIT for the dynamic content to load. 
    # This halts execution until a standard <tr> (table row) appears on the page, timing out after 20 seconds.
    wait = WebDriverWait(driver, 20)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "tr")))
    
    # Give it an extra 3 seconds just to make sure all data fields populate
    time.sleep(3)
    
    # 2. Extract elements
    table_rows = driver.find_elements(By.TAG_NAME, "tr")
    scraped_data = []
    
    print(f"Found {len(table_rows)} total rows (including headers). Parsing data...")
    
    for row in table_rows:
        cells = row.find_elements(By.TAG_NAME, "td")
        if cells:
            row_data = [cell.text.strip() for cell in cells]
            scraped_data.append(row_data)
            
    # 3. Export to CSV
    if scraped_data:
        df = pd.DataFrame(scraped_data)
        df.to_csv("ecsc_census_dynamic_data.csv", index=False)
        print(f"Success! Exported {len(scraped_data)} rows to 'ecsc_census_dynamic_data.csv'.")
    else:
        print("Table rows found, but no data cells ('td') inside them. Double check page architecture.")

except Exception as e:
    print(f"An error occurred or loading timed out: {e}")

finally:
    # Always close the background browser session
    driver.quit()


In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

url = "https://edv.ecsc.gov.et/ICSMISCensus"

# Setup options
options = webdriver.ChromeOptions()
# 1. REMOVED --headless so you can visually see if the page blocks or loads slow
options.add_argument("--start-maximized")
options.add_argument("--disable-gpu")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

try:
    print("Opening browser window...")
    driver.get(url)
    
    # 2. INCREASED wait window to 45 seconds to accommodate slow server response times
    print("Waiting for page assets and table data to render...")
    wait = WebDriverWait(driver, 45)
    
    # Wait until ANY table row or generic content body element appears
    wait.until(EC.presence_of_element_located((By.XPATH, "//tr | //table | //div[contains(@class, 'content')]")))
    
    # Give the page 5 solid seconds to settle and paint the UI
    time.sleep(5)
    
    # 3. Pull all table rows
    table_rows = driver.find_elements(By.TAG_NAME, "tr")
    scraped_data = []
    
    print(f"Elements loaded successfully! Found {len(table_rows)} rows to parse.")
    
    for row in table_rows:
        cells = row.find_elements(By.TAG_NAME, "td")
        if cells:
            row_data = [cell.text.strip() for cell in cells]
            scraped_data.append(row_data)
        else:
            # Fallback: if structure is div-based, get raw text lines
            text_line = row.text.strip()
            if text_line:
                scraped_data.append([text_line])
            
    if scraped_data:
        df = pd.DataFrame(scraped_data)
        df.to_csv("ecsc_census_debugged.csv", index=False)
        print("Success! Data written to 'ecsc_census_debugged.csv'.")
    else:
        print("Browser loaded the structure, but no text or row elements were extracted.")

except Exception as e:
    print(f"\nExecution failed with error: {e}")
    print("\n--- Troubleshooting Step ---")
    print("Look at the opened automated Chrome browser window right now.")
    print("Is it displaying a login page, a blank white screen, or an 'Access Denied' error?")

finally:
    # Keeps browser open for 10 seconds so you can physically read what's on screen before closing
    time.sleep(10)
    driver.quit()


Opening browser window...
Waiting for page assets and table data to render...

Execution failed with error: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=148.0.7778.217)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff692cf7de5+14895]
	chromedriver!GetHandleVerifier [0x7ff692cf7e50+14900]
	chromedriver!(No symbol) [0x7ff692a5d5ad]
	chromedriver!(No symbol) [0x7ff692a340f2]
	chromedriver!(No symbol) [0x7ff692ae6f56]
	chromedriver!(No symbol) [0x7ff692b04172]
	chromedriver!(No symbol) [0x7ff692aa9df8]
	chromedriver!(No symbol) [0x7ff692aaace3]
	chromedriver!GetHandleVerifier [0x7ff69300cc49+3296f9]
	chromedriver!GetHandleVerifier [0x7ff693007375+323e25]
	chromedriver!GetHandleVerifier [0x7ff69302bc82+348732]
	chromedriver!GetHandleVerifier [0x7ff692d16045+32af5]
	chromedriver!GetHandleVerifier [0x7ff692d1ecec+3b79c]
	chromedriver!GetHandleVerifier [0x7ff692d01bc4+1e674]
	chromedriver!GetHandleVerifier [0x7ff692d01d5